In [13]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict

# 加载 .env 文件中的环境变量
load_dotenv()

class HelloAgentsLLM:
    """
    LLM客户端
    """
    def __init__(self, model: str = None, apikey: str = None, baseUrl: str = None, timeout: int = None) -> None:
        """
        初始化LLM客户端
        """
        self.model = model or os.getenv("DASHSCOPE_MODEL")
        self.apikey = apikey or os.getenv("DASHSCOPE_API_KEY")
        self.baseUrl = baseUrl or os.getenv("DASHSCOPE_BASE_URL")
        self.timeout = timeout or int(os.getenv("DASHSCOPE_TIMEOUT", 60))
        # 校验必要参数
        if not all([self.model, self.apikey, self.baseUrl]):
            raise ValueError("模型、API密钥和基础URL不能为空")

        self.client = OpenAI(
            api_key=self.apikey,
            base_url=self.baseUrl,
            timeout=self.timeout,
        )

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        调用LLM模型
        """
        print(f"正在调用{self.model}模型...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True
            )
            # 处理流式响应
            print("大语言模型响应成功")
            collection_cotent = []
            for chunk in response:
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collection_cotent.append(content)
            print()
            return "".join(collection_cotent)
        
        except Exception as e:
            print(f"调用模型{self.model}时出错: {e}")
            return ""

In [31]:
from hello import client
from serpapi import SerpApiClient

def search(query: str) -> str:
    """
    调用SerpApi搜索
    """
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            raise ValueError("SERPAPI_API_KEY 环境变量未设置")

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "hl": "zh-CN",
            "gl": "cn",
            "num": 3,
        }

        client = SerpApiClient(params)
        results = client.get_dict()

        # 提取搜索结果
        organic_results = results.get("organic_results", [])
        snippets = [result.get("snippet", "") for result in organic_results]
        return "\n".join(snippets)
    except Exception as e:
        print(f"调用SerpApi搜索时出错: {e}")
        return ""
    
if __name__ == "__main__":
    query = "iran"
    results = serch(query)
    print(results)

Iran he Â-chû yit-ke koet-kâ. Iran Yî-sṳ̂-làn Khiung-fò-koet. جمهوری اسلامی ایران. Iran khì. koet-khì. Iran koet-fî. koet-fî. Kiet-ngièn: استقلال، آزادی، ...
Latest Iran News. Premier source of in-depth news, analysis, insights, and opinions on Iran by native and non-native journalists and experts.


In [ ]:
from typing import List, Dict

class ToolExecutor:
    """
    工具执行器
    """
    def __init__(self):
        """
        初始化工具执行器
        """
        self.tools: Dict[str, Dict[str, any]] = {}

    def registerTool(self, tool_name: str, description: str, function: any):
        """
        注册工具
        """
        self.tools[tool_name] = {"description": description, "function": function}
        print(f"工具 {tool_name} 已注册")

    def getTool(self, tool_name: str) -> callable:
        """
        获取工具
        """
        return self.tools.get(tool_name, {}).get("function", None)

    def getAvailableTools(self) -> str:
        """
        获取所有可用工具的名称和描述
        """
        return "\n".join([
            f"- {tool_name}: {tool_info['description']}\n"
            for tool_name, tool_info in self.tools.items()
        ])

    

In [19]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下：
{available_tools}

严格按照以下格式进行回应：
Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一：
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你确定问题的答案时，使用此操作。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用`Finish[最终答案]`来输出最终答案。

现在，请解决以下问题：
Question: {user_question}
History: {history}
"""

In [35]:
import re

class ReActAgent:
    """
    ReAct智能体
    """
    def __init__(self, llm_client: HelloAgentsLLM, tool_executor: ToolExecutor, max_steps: int=5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []
    
    def run(self, user_question: str):
        """
        运行ReAct智能体
        """
        self.history = [] # 清空历史记录

        for step in range(self.max_steps):
            print(f"第{step}步")

            # 构建提示词
            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                available_tools=tools_desc,
                user_question=user_question,
                history=history_str
            )
            print(f"\n提示词: {prompt}\n")

            # 调用LLM模型
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)

            if not response_text:
                print("错误：LLM模型返回空响应")
                break
        
            # 解析LLM模型输出
            thought, action = self._parse_output(response_text)

            if thought:
                print(f"Thought: {thought}")

            if not action:
                print("警告：未解析到有效Action，流程终止")
                break

            # 4.执行Action
            if action.startswith("Finish"):
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"Final Answer: {final_answer}")
                return final_answer
            
            # 5.执行Tool
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                # 跳过无效Action
                continue

            print(f"Tool: {tool_name}, Input: {tool_input}")

            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                print(f"警告：工具{tool_name}不存在")
                continue
            else:
                # 执行Tool
                observation = tool_function(tool_input)
            
            print(f"Observation: {observation}")
            
            # 6.更新历史记录
            self.history.append(f"Action: {action}\nObservation: {observation}")
        
        # 循环结束
        print("ReAct智能体运行完成")
        return None
        
    def _parse_output(self, text: str):
        """
        解析LLM模型的输出，提取Thought和Action
        """
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        """
        解析Action文本，提取工具名称和参数
        """
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        else:
            return None, None




In [36]:
# 测试ReAct智能体
# 1. 初始化工具执行器
toolExecutor = ToolExecutor()

# 2. 注册我们的实战搜索工具
search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
toolExecutor.registerTool("Search", search_description, search)

agent = ReActAgent(HelloAgentsLLM(), toolExecutor)
agent.run(user_question="长沙天气怎么样")


工具 Search 已注册
第0步

提示词: 
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下：
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。


严格按照以下格式进行回应：
Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一：
- `{tool_name}[{tool_input}]`:调用一个可用工具。
- `Finish[最终答案]`:当你确定问题的答案时，使用此操作。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用`Finish[最终答案]`来输出最终答案。

现在，请解决以下问题：
Question: 长沙天气怎么样
History: 


正在调用qwen-plus模型...
大语言模型响应成功
Thought: 我需要获取长沙当前的天气信息。由于天气是实时变化的，我的知识库中没有最新数据，因此应使用搜索引擎查询当前长沙的天气情况。
Action: Search[长沙当前天气]
Thought: 我需要获取长沙当前的天气信息。由于天气是实时变化的，我的知识库中没有最新数据，因此应使用搜索引擎查询当前长沙的天气情况。
Tool: Search, Input: 长沙当前天气
Observation: 4日（今天）. 多云. 7℃. <3级 · 5日（明天）. 多云. 17℃/7℃. <3级 · 6日（后天）. 小雨转晴. 17℃/7℃. <3级 · 7日（周六）. 多云. 18℃/8℃. <3级 · 8日（周日）. 小雨. 12℃/8℃. <3级 ...
长沙市. 6.8℃. 日落18:30. 降水量. 0mm. 西北风. 微风. 相对湿度. 99%. 体感温度. 5.6℃. 空气质量：-. 舒适度：凉，不舒适. 雷达图. 24小时预报7天预报10天预报11-30天 ...
无明显降温，感冒机率较低。 ... 天气凉，在户外运动请注意增减衣物。 ... 建议着厚外套加毛衣等服装。 ... 无雨且风力较小，易保持清洁度。 ... 涂擦SPF大于15、PA+防晒护肤品。
第1步

提示词: 
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下：
- Se

'长沙当前天气：多云，气温6.8℃，体感温度5.6℃，西北风微风，相对湿度99%，舒适度为“凉，不舒适”，建议穿厚外套加毛衣；未来几日以多云为主，6日有小雨转晴，8日有小雨，整体无明显降温，感冒风险较低。'